# PyTorch Lightning GPU & TPU Trainer with KFolds and F1 Loss + W&B logging ⚡️

To run the model on TPU, un-comment and run the below cell and change the `gpus=1` argument to `tpu_cores=1` or `tpu_cores=8` in the `Trainer` object at the bottom of the notebook.

You can also extend the notebook to use Multi-GPU setting by changing the current `gpus=1` to `gpus=2` when using the 2x T4 GPUs.
    
I am training a `swin_base_patch4_window7_224` model with `img_size=224` on 1x P100 GPU. As indicated above, you can always perform minimal changes in this training code to extend that to single or mulitple GPUs or TPU cores.

**Feel free to fork and change the models and do some preprocessing, but if you do please leave an upvote :)**

<center>
<img src="https://img.shields.io/badge/Upvote-If%20you%20like%20my%20work-07b3c8?style=for-the-badge&logo=kaggle">
</center>

Uncomment this cell to run the notebook with TPUs 👇🏻

In [ ]:
# %%capture
# ! curl https://raw.githubusercontent.com/pytorch/xla/master/contrib/scripts/env-setup.py -o pytorch-xla-env-setup.py
# ! python pytorch-xla-env-setup.py --version 1.7 --apt-packages libomp5 libopenblas-dev
# ! python pytorch-xla-env-setup.py --apt-packages libomp5 libopenblas-dev
# !pip install cloud-tpu-client==0.10 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.7-cp36-cp36m-linux_x86_64.whl
# !pip install cloud-tpu-client==0.10 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

## Installation and Imports

In [ ]:
%%capture
!pip install timm
!pip install einops
!apt-get update && apt-get install -y python3-opencv
!pip install -q opencv-python
!pip install albumentations
!pip install --upgrade wandb
!pip install --upgrade torchmetrics
!pip install --upgrade pytorch-lightning

In [ ]:
import torch_xla

In [ ]:
import os
import sys
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import wandb
from einops import rearrange

import timm
import torch
import transformers
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset

import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

from sklearn.model_selection import StratifiedGroupKFold

from albumentations import (
    HorizontalFlip, VerticalFlip, Flip, OneOf, 
    Compose, Normalize,Resize
)
from albumentations.pytorch import ToTensorV2

import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
os.environ.pop('TPU_PROCESS_ADDRESSES')
os.environ.pop('CLOUD_TPU_TASK_ID')
os.environ["NUMBA_NUM_THREADS"] = "1"

## Config File and Wandb

In [ ]:
Config = {
    'TRAIN_BS': 16,
    'VALID_BS': 16,
    'IMG_SIZE': (224, 224),
    'MODEL_NAME': 'swin_base_patch4_window7_224',
    'NUM_WORKERS': 8,
    'PARENT_PATH': '/kaggle/input/rsna-mammography-images-as-pngs/images_as_pngs_cv2_384/train_images_processed_cv2_384/',
    'FILE_PATH': '/kaggle/input/rsna-breast-cancer-detection/train.csv',
    'LOSS': 'BCEWithLogitsLoss',
    'EVAL_METRIC': 'F1',
    'NB_EPOCHS': 2,
    'SPLITS': 5,
    'min_lr': 1e-6,
    'T_max': 20,
    'T_0': 25,
    'fc_dropout': 0.2,
    'betas': (0.9, 0.999),
    'NUM_LABELS': 1,
    'LR': 2e-4,
    'competition': 'rsna_mammography',
    '_wandb_kernel': 'tanaym',
}

### About W&B:
<center><img src="https://i.imgur.com/gb6B4ig.png" width="400" alt="Weights & Biases"/></center><br>
<p style="text-align:center">WandB is a developer tool for companies turn deep learning research projects into deployed software by helping teams track their models, visualize model performance and easily automate training and improving models.
We will use their tools to log hyperparameters and output metrics from your runs, then visualize and compare results and quickly share findings with your colleagues.<br><br></p>

To login to W&B, you can use below snippet.

```python
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wb_key)
```
Make sure you have your W&B key stored as `WANDB_API_KEY` under Add-ons -> Secrets

You can view [this](https://www.kaggle.com/ayuraj/experiment-tracking-with-weights-and-biases) notebook to learn more about W&B tracking.

If you don't want to login to W&B, the kernel will still work and log everything to W&B in anonymous mode.

In [ ]:
# Start W&B logging
# W&B Login
from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# wb_key = user_secrets.get_secret("WANDB_API_KEY")

# wandb.login(key=wb_key)

# run = wandb.init(
#     project='pytorch_lightning',
#     config=Config,
#     group='vision',
#     job_type='train',
# )

In [ ]:
# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=0, alpha=None, size_average=True):
        """
        Focal Loss function class taken from:
        https://github.com/clcarwin/focal_loss_pytorch
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        if isinstance(alpha,(float,int)): self.alpha = torch.Tensor([alpha,1-alpha])
        if isinstance(alpha,list): self.alpha = torch.Tensor(alpha)
        self.size_average = size_average

    def forward(self, input, target):
        if input.dim()>2:
            input = input.view(input.size(0),input.size(1),-1)  # N,C,H,W => N,C,H*W
            input = input.transpose(1,2)    # N,C,H*W => N,H*W,C
            input = input.contiguous().view(-1,input.size(2))   # N,H*W,C => N*H*W,C
        target = target.view(-1,1)

        logpt = F.log_softmax(input)
        logpt = logpt.gather(1,target)
        logpt = logpt.view(-1)
        pt = Variable(logpt.data.exp())

        if self.alpha is not None:
            if self.alpha.type()!=input.data.type():
                self.alpha = self.alpha.type_as(input.data)
            at = self.alpha.gather(0,target.data.view(-1))
            logpt = logpt * Variable(at)

        loss = -1 * (1-pt)**self.gamma * logpt
        if self.size_average: return loss.mean()
        else: return loss.sum()
        
def probabilistic_f1(labels, preds, beta=1):
    """
    Function taken from Awsaf's notebook:
    https://www.kaggle.com/code/awsaf49/rsna-bcd-efficientnet-tf-tpu-1vm-train
    """
    eps = 1e-5
    preds = preds.clip(0, 1)
    y_true_count = labels.sum()
    ctp = preds[labels==1].sum()
    cfp = preds[labels==0].sum()
    beta_squared = beta * beta
    c_precision = ctp / (ctp + cfp + eps)
    c_recall = ctp / (y_true_count + eps)
    if (c_precision > 0 and c_recall > 0):
        result = (1 + beta_squared) * (c_precision * c_recall) / (beta_squared * c_precision + c_recall + eps)
        return result
    else:
        return 0.0
    
def wandb_log(**kwargs):
    for k, v in kwargs.items():
        wandb.log({k: v})

## Dataset Class - for loading in data

In [ ]:
class RSNAData(Dataset):
    def __init__(self, df, img_folder, augments=None, is_test=False):
        self.df = df
        self.is_test = is_test
        self.augments = augments
        self.img_folder = img_folder
        
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_folder, self.df['img_name'][idx])
        img = cv2.imread(img_path)
        img = cv2.resize(img, Config['IMG_SIZE'])
        if self.augments:
            img = self.augments(image=img)['image']
        img = torch.tensor(img, dtype=torch.float)
        # Rearrange the image dimensions so that channels are first in format
        # This is because Swin Transformer Model requires Channels (c) to come first
        img = rearrange(img, 'h w c -> c h w')
        
        if not self.is_test:
            target = self.df['cancer'][idx]
            target = torch.tensor(target, dtype=torch.float)
            return (img, target)
        return (img)
    
    def __len__(self):
        return len(self.df)

## Basic Image Augmentations

In [ ]:
class Augments:
    """
    Contains Train, Validation Augments
    """
    train_augments = Compose([
        Resize(*Config['IMG_SIZE'], p=1.0),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0
        ),
        ToTensorV2(p=1.0),
    ],p=1.)
    
    valid_augments = Compose([
        Resize(*Config['IMG_SIZE'], p=1.0),
        Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0
        ),
        ToTensorV2(p=1.0),
    ], p=1.)

## Model Class using `pl.LightningModule` ⚡️

In [ ]:
class MyNet(pl.LightningModule):
    def __init__(self, model_name, num_labels, pretrained):
        super(MyNet, self).__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)
        self.n_features = self.model.head.in_features
        self.model.reset_classifier(0)
        self.fc = nn.Linear(self.n_features, num_labels)
    def forward(self, x):
        features = self.model(x)
        out = self.fc(features)
        return out

In [ ]:
class RSNAModel(pl.LightningModule):
    def __init__(self, pretrained=True):
        super(RSNAModel, self).__init__()
        # Model Architecture
#         self.model = timm.create_model(Config['MODEL_NAME'], pretrained=pretrained)
#         self.n_features = self.model.head.in_features
#         self.model.reset_classifier(0)
#         self.fc = nn.Linear(self.n_features, Config['NUM_LABELS'])
        self.model = MyNet(Config['MODEL_NAME'], Config['NUM_LABELS'], pretrained)
        # Loss functions
        self.train_loss = nn.BCEWithLogitsLoss()
        self.valid_loss = nn.BCEWithLogitsLoss()

        # Metric
        self.f1 = torchmetrics.F1Score(task='binary')
        
    def forward(self, x):
#         features = self.model(x)
#         output = self.fc(features)
        output = self.model(x)
        return output
    
    def training_step(self, batch, batch_idx):
        imgs = batch[0]
        target = batch[1]
        
        out = self(imgs).view(-1)
        train_loss = self.train_loss(out, target)
        
        logs = {'train_loss': train_loss}
#         wandb_log(train_step_loss=train_loss.item())
        return {'loss': train_loss, 'log': logs}
    
    def validation_step(self, batch, batch_idx):
        imgs = batch[0]
        target = batch[1]
        
        out = self(imgs).view(-1)
        valid_loss = self.valid_loss(out, target)
        
        self.f1(out, target)
        f1_current = self.f1(out, target)
        self.log('f1_valid_epoch', self.f1, on_epoch=True, on_step=True)
        
#         wandb_log(val_step_loss=valid_loss.item(), f1_step_valid=f1_current)
        
        return {'val_loss': valid_loss, 'f1_score': f1_current}
    
    def validation_end(self, outputs):
        avg_loss = torch.stack([x['val_loss'] for x in outputs]).mean()
        
        logs = {'val_loss': avg_loss}
#         wandb_log(val_loss=avg_loss)
        
        print(f"val_loss: {avg_loss}")
        return {'avg_val_loss': avg_loss, 'log': logs}
    
    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=Config['LR'])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, 
            T_max=Config['T_max'],
            eta_min=Config['min_lr']
        )
        
        return [opt], [sch]

## CSV File loading and Model Training

In [ ]:
# Load the data and pass it onto the training function
df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")
df['img_name'] = df['patient_id'].astype(str) + "/" + df['image_id'].astype(str) + ".png"
df = df.sample(frac=1).reset_index(drop=True)
df.head()
df=df[:1000]

In [ ]:
if __name__ == "__main__":
    kfold = StratifiedGroupKFold(n_splits=Config['SPLITS'])
    for fold_, (train_idx, valid_idx) in enumerate(kfold.split(df, df['cancer'].values, df['patient_id'].values)):
        print(f"{'='*40} Fold: {fold_} / 5 {'='*40}")
        
        train_df = df.loc[train_idx].reset_index(drop=True)
        valid_df = df.loc[valid_idx].reset_index(drop=True)
        
        train_dataset = RSNAData(
            df = train_df,
            img_folder = Config['PARENT_PATH']
        )
        valid_dataset = RSNAData(
            df = valid_df,
            img_folder = Config['PARENT_PATH'],
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=Config['TRAIN_BS'],
            shuffle=True,
            num_workers=Config['NUM_WORKERS'],
        )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=Config['VALID_BS'],
            shuffle=False,
            num_workers=Config['NUM_WORKERS'],
        )

        model = RSNAModel()
        trainer = pl.Trainer(
            max_epochs=Config['NB_EPOCHS'],
            accelerator='auto',
            devices='auto',
            deterministic=True,
        )
        trainer.fit(model, train_loader, valid_loader)

In [ ]:
# # Finish the logging run
# run.finish()

<center>
<img src="https://img.shields.io/badge/Upvote-If%20you%20like%20my%20work-07b3c8?style=for-the-badge&logo=kaggle">
</center>